# Phase 1a: Baseline - Vector RAG (DPR / Contriever)

**Muc tieu:** Implement va do baseline Vector-based RAG de so sanh voi GraphRAG methods.

**Pipeline:**
1. Encode passages/triples thanh vectors (DPR hoac Contriever)
2. Encode query
3. Retrieve top-k passages bang cosine similarity
4. Feed retrieved context + question vao LLM
5. Do F1, EM

**Tai sao can baseline nay:**
Cho thay GraphRAG vuot troi hon "flat" retrieval o nhung cau hoi multi-hop.

In [ ]:
# === Setup ===
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.insert(0, '/content/drive/MyDrive/GraphRAG_Research/utils')
from common import *

In [ ]:
%%capture
!pip install sentence-transformers faiss-gpu transformers datasets

## 1. Load Dataset

In [ ]:
import json
import numpy as np

# --- HotpotQA ---
HOTPOTQA_DEV = os.path.join(DATASET_DIR, 'hotpotqa', 'hotpot_dev_distractor_v1.json')
with open(HOTPOTQA_DEV) as f:
    hotpot_data = json.load(f)

print(f'HotpotQA dev: {len(hotpot_data)} samples')

# Chuan bi passages tu context
def extract_passages(sample):
    """Extract passages tu HotpotQA context."""
    passages = []
    for title, sentences in sample['context']:
        text = ' '.join(sentences)
        passages.append({'title': title, 'text': text})
    return passages

# Test
sample = hotpot_data[0]
passages = extract_passages(sample)
print(f'\nSample question: {sample["question"]}')
print(f'Answer: {sample["answer"]}')
print(f'Num passages: {len(passages)}')

## 2. Build Retriever

Dung Contriever (Facebook) - dense retriever khong can fine-tune tren specific dataset.
Thay bang DPR neu muon fine-tuned version.

In [ ]:
from sentence_transformers import SentenceTransformer
import faiss
import torch


class VectorRetriever:
    """Dense passage retriever dung sentence-transformers + FAISS."""
    
    def __init__(self, model_name='facebook/contriever'):
        self.encoder = SentenceTransformer(model_name)
        self.index = None
        self.passages = []
    
    def index_passages(self, passages, batch_size=64):
        """Encode va index passages."""
        self.passages = passages
        texts = [p['text'] for p in passages]
        
        print(f'Encoding {len(texts)} passages...')
        embeddings = self.encoder.encode(texts, batch_size=batch_size,
                                         show_progress_bar=True)
        embeddings = embeddings.astype('float32')
        
        # FAISS index
        dim = embeddings.shape[1]
        self.index = faiss.IndexFlatIP(dim)  # Inner product (cosine after norm)
        faiss.normalize_L2(embeddings)
        self.index.add(embeddings)
        print(f'Indexed {self.index.ntotal} passages, dim={dim}')
    
    def retrieve(self, query, top_k=5):
        """Retrieve top-k passages cho 1 query."""
        q_emb = self.encoder.encode([query]).astype('float32')
        faiss.normalize_L2(q_emb)
        scores, indices = self.index.search(q_emb, top_k)
        results = []
        for score, idx in zip(scores[0], indices[0]):
            results.append({
                'passage': self.passages[idx],
                'score': float(score)
            })
        return results


retriever = VectorRetriever('facebook/contriever')
print('Retriever initialized.')

## 3. RAG Pipeline: Retrieve + Generate

In [ ]:
import openai

# === CAU HINH LLM ===
# Chon 1 trong cac option:
# Option 1: OpenAI API
# openai.api_key = 'YOUR_API_KEY'
# LLM_MODEL = 'gpt-3.5-turbo'

# Option 2: Local model (Flan-T5, LLaMA via HuggingFace)
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

LLM_NAME = 'google/flan-t5-large'  # Chay duoc tren Colab, khong can API
tokenizer = AutoTokenizer.from_pretrained(LLM_NAME)
llm_model = AutoModelForSeq2SeqLM.from_pretrained(LLM_NAME).to('cuda')
print(f'LLM loaded: {LLM_NAME}')


def generate_answer(question, context, max_length=128):
    """Generate answer tu question + retrieved context."""
    prompt = f"""Answer the question based on the given context.

Context:
{context}

Question: {question}
Answer:"""
    inputs = tokenizer(prompt, return_tensors='pt', max_length=1024,
                       truncation=True).to('cuda')
    outputs = llm_model.generate(**inputs, max_length=max_length)
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return answer

## 4. Evaluate tren HotpotQA

In [ ]:
from tqdm import tqdm

def evaluate_vector_rag(data, retriever, top_k=5, max_samples=500):
    """Evaluate Vector RAG pipeline tren dataset."""
    predictions = []
    ground_truths = []
    retrieval_costs = []
    
    for sample in tqdm(data[:max_samples], desc='Evaluating'):
        question = sample['question']
        answer = sample['answer']
        
        # Index passages cho sample nay
        passages = extract_passages(sample)
        retriever.index_passages(passages)
        
        # Retrieve
        retrieved = retriever.retrieve(question, top_k=top_k)
        context = '\n'.join([r['passage']['text'] for r in retrieved])
        retrieval_costs.append(top_k)
        
        # Generate
        pred = generate_answer(question, context)
        predictions.append(pred)
        ground_truths.append(answer)
    
    # Compute metrics
    results = evaluate_batch(predictions, ground_truths, metrics=['f1', 'em'])
    results['avg_retrieval_cost'] = np.mean(retrieval_costs)
    results['num_samples'] = len(predictions)
    return results, predictions, ground_truths


# === RUN EVALUATION ===
# Bat dau voi so luong nho de test pipeline
results_k3, preds_k3, gts_k3 = evaluate_vector_rag(
    hotpot_data, retriever, top_k=3, max_samples=100
)
print(f'\n=== Vector RAG (top-3) ===')
print(f'F1: {results_k3["f1"]:.4f}')
print(f'EM: {results_k3["em"]:.4f}')

In [ ]:
# === So sanh top-k khac nhau ===
results_all = {}
for k in [1, 3, 5, 10]:
    results, _, _ = evaluate_vector_rag(
        hotpot_data, retriever, top_k=k, max_samples=100
    )
    results_all[f'top_{k}'] = results
    print(f'top-{k}: F1={results["f1"]:.4f}, EM={results["em"]:.4f}')

# Save results
save_results(results_all, 'baseline_vector_rag_hotpotqa')

## 5. Error Analysis

Phan tich cac truong hop Vector RAG fail -> Motivation cho GraphRAG

In [ ]:
# Tim cac truong hop F1 = 0 (fail hoan toan)
failures = []
for i, (pred, gt) in enumerate(zip(preds_k3, gts_k3)):
    f1 = compute_f1(pred, gt)
    if f1 == 0:
        failures.append({
            'idx': i,
            'question': hotpot_data[i]['question'],
            'gold_answer': gt,
            'predicted': pred,
            'type': hotpot_data[i]['type'],  # bridge or comparison
            'level': hotpot_data[i]['level'],
        })

print(f'Total failures: {len(failures)}/{len(preds_k3)}')
print(f'\n--- Failure examples (multi-hop) ---')
for f in failures[:5]:
    print(f'Q: {f["question"]}')
    print(f'Gold: {f["gold_answer"]}')
    print(f'Pred: {f["predicted"]}')
    print(f'Type: {f["type"]}, Level: {f["level"]}')
    print('---')

## Ket luan Phase 1a

- Vector RAG chi dung semantic similarity -> kho voi multi-hop questions
- Cac cau hoi 'bridge' type (can 2+ facts) thuong fail
- => Motivation: can structured retrieval (GraphRAG) de follow reasoning chains